# PySP: Pyomo Stochastic Programming - Tutorial Completo

## 1. Introdução ao PySP

**PySP** (Pyomo Stochastic Programming) é um módulo específico dentro do Pyomo para **Programação Estocástica**. Ele implementa algoritmos de **decomposição** (como L-shaped e Progressive Hedging) para resolver problemas de otimização sob incerteza.

### O que é Programação Estocástica?
- Otimização com parâmetros incertos
- Decisões em múltiplos estágios
- Cenários para representar a incerteza
- **Decisões aqui-e-agora** vs **decisions wait-and-see**

### Tipos de Problemas que PySP aborda:
1. **Two-stage stochastic programming** (2-SP)
2. **Multi-stage stochastic programming** (M-SP)
3. **Chance-constrained programming**
4. **Robust optimization** (até certo ponto)
5. **Stochastic dynamic programming**

## 2. Instalação e Configuração

```bash
# Instalar Pyomo e PySP
pip install pyomo
pip install pyomo.extras  # Para funcionalidades adicionais
pip install ply  # Necessário para parsing

# Para solvers
pip install glpk
pip install ipopt  # Para NLP estocástico
pip install couenne  # Para MINLP estocástico

# Verificar instalação do PySP
python -c "import pyomo.pysp; print('PySP instalado com sucesso')"
```

## 3. Arquitetura Geral do PySP

```
┌─────────────────────────────────────────────────┐
│               Reference Model                   │
│  (todas variáveis, parâmetros e restrições)    │
└─────────────────────────────────────────────────┘
                    │
          ┌─────────┴─────────┐
          │                   │
┌─────────────────┐ ┌─────────────────┐
│ Scenario Tree   │ │ Scenario Data   │
│ (estrutura)     │ │ (valores)       │
└─────────────────┘ └─────────────────┘
                    │
          ┌─────────┴─────────┐
          │                   │
┌─────────────────┐ ┌─────────────────┐
│ Decomposition   │ │ Solver Manager  │
│ Algorithm       │ │ (coordenação)   │
└─────────────────┘ └─────────────────┘
```

## 4. Componentes Principais

### 4.1 Estrutura de Arquivos
```
projeto_estocastico/
├── ReferenceModel.py      # Modelo determinístico base
├── ScenarioStructure.dat  # Árvore de cenários
├── ScenarioData/          # Dados por cenário
│   ├── Scenario1.dat
│   ├── Scenario2.dat
│   └── ...
├── runph.py              # Script para Progressive Hedging
└── solve.py              # Script para resolver
```

### 4.2 Reference Model (Modelo de Referência)
```python
# ReferenceModel.py
from pyomo.environ import *

def create_model():
    model = ConcreteModel()
    
    # Primeiro estágio (aqui-e-agora)
    model.x1 = Var(within=NonNegativeReals)  # Decisão antes da incerteza
    model.x2 = Var(within=NonNegativeReals)
    
    # Parâmetros estocásticos (serão sobrescritos por cenário)
    model.demand = Param(initialize=0, mutable=True)
    model.cost = Param(initialize=0, mutable=True)
    
    # Segundo estágio (wait-and-see) - definido no cenário
    # A variável y será criada em cada cenário
    
    # Restrições do primeiro estágio
    model.first_stage_constraint = Constraint(
        expr = model.x1 + model.x2 <= 100
    )
    
    return model

def create_instance(model, scenario_name=None):
    return model.clone()
```

### 4.3 Scenario Structure (Estrutura de Cenários)
```python
# ScenarioStructure.dat
set Stages := FirstStage SecondStage;
set Nodes := RootNode Node1 Node2 Node3;

param NodeStage :=
    RootNode    FirstStage
    Node1       SecondStage
    Node2       SecondStage
    Node3       SecondStage;

set Children[RootNode] := Node1 Node2 Node3;

param ConditionalProbability :=
    Node1       0.3
    Node2       0.5
    Node3       0.2;

set Scenarios := Scenario1 Scenario2 Scenario3;

param ScenarioLeafNode :=
    Scenario1   Node1
    Scenario2   Node2
    Scenario3   Node3;

param StageCostVariable :=
    FirstStage    None
    SecondStage   SecondStageCost;
```

### 4.4 Scenario Data (Dados por Cenário)
```python
# ScenarioData/Scenario1.dat
param demand := 80;
param cost := 5;

var y;
subject to SecondStageConstraint:
    y <= demand;
subject to LinkingConstraint:
    y <= x1 + x2;
minimize SecondStageCost:
    cost * y;
```

## 5. Algoritmos de Decomposição

### 5.1 Progressive Hedging (PH) - Para 2 estágios
```python
# runph.py
from pyomo.pysp.phutils import scenario_tree
from pyomo.pysp import ph
import pyomo.environ as pyo

# 1. Configurar PH
ph_options = {
    '--solver': 'glpk',
    '--verbose': True,
    '--default-rho': 1.0,
    '--max-iterations': 100,
    '--linearize-nonbinary-penalty-terms': True,
    '--disable-weighted-average': False,
}

# 2. Criar PH solver
ph = ph.PH(
    options=ph_options,
    scenario_tree=None  # Será criado automaticamente
)

# 3. Resolver
ph.solve()
```

### 5.2 L-Shaped (Benders Decomposition) - Para problemas lineares
```python
# runlshaped.py
from pyomo.pysp.lshaped import LShapedMethod

lshaped_options = {
    '--solver': 'cplex',
    '--verbose': True,
    '--max-iterations': 50,
    '--tee': True,
    '--check-bound-improvement-frequency': 5,
}

lshaped = LShapedMethod(options=lshaped_options)
lshaped.solve()
```

## 6. Exemplos Completos

### Exemplo 1: Problema do Newsvendor (Vendedor de Jornais)
```python
# newsvendor.py
from pyomo.environ import *
from pyomo.pysp import *
import numpy as np

# Modelo de referência
def create_reference_model():
    model = ConcreteModel()
    
    # Primeiro estágio: quantos jornais comprar
    model.order_quantity = Var(within=NonNegativeIntegers)
    
    # Parâmetros estocásticos
    model.demand = Param(initialize=0, mutable=True)
    model.buy_price = Param(initialize=0.5, mutable=False)
    model.sell_price = Param(initialize=1.0, mutable=False)
    model.salvage_price = Param(initialize=0.2, mutable=False)
    
    # Função objetivo (será estendida por cenário)
    model.FirstStageCost = Expression(
        expr = model.buy_price * model.order_quantity
    )
    
    return model

# Gerador de cenários
def generate_scenarios(num_scenarios=10):
    scenarios = []
    for i in range(num_scenarios):
        demand = np.random.poisson(lam=100)  # Demanda Poisson
        scenario_data = {
            'demand': demand,
            'probability': 1.0/num_scenarios
        }
        scenarios.append(scenario_data)
    return scenarios

# Criar estrutura de cenários dinamicamente
def create_scenario_tree(scenarios):
    from pyomo.pysp.scenariotree.tree_structure_model import \
        CreateConcreteTwoStageScenarioTreeModel
    
    st_model = CreateConcreteTwoStageScenarioTreeModel()
    
    # Primeiro estágio
    st_model.Stages.add('FirstStage')
    st_model.Stages.add('SecondStage')
    
    # Nós
    st_model.Nodes.add('RootNode')
    st_model.Nodes.add('LeafNode1')
    
    # Atributos dos nós
    st_model.NodeStage['RootNode'] = 'FirstStage'
    st_model.NodeStage['LeafNode1'] = 'SecondStage'
    
    st_model.Children['RootNode'].add('LeafNode1')
    st_model.ConditionalProbability['LeafNode1'] = 1.0
    
    return st_model

# Script principal
if __name__ == '__main__':
    # Criar cenários
    scenarios = generate_scenarios(100)
    
    # Configurar PH
    from pyomo.pysp.phutils import PHUtils
    
    utils = PHUtils()
    
    # Configurar opções
    options = {
        '--solver': 'glpk',
        '--max-iterations': 50,
        '--default-rho': 10.0,
        '--scenario-tree-seed': 12345,
    }
    
    # Resolver usando PH
    import pyomo.pysp.phsolverserver as phsolverserver
    
    with phsolverserver.PHSolverServer(options=options) as ph:
        results = ph.solve()
        
        print("Solução ótima encontrada!")
        print(f"Quantidade a pedir: {results.xstar['order_quantity']}")
        print(f"Custo esperado: {results.objval}")
```

### Exemplo 2: Planejamento de Energia com Demanda Incerta
```python
# energy_planning.py
from pyomo.environ import *
from pyomo.pysp import *
import pandas as pd

def create_energy_model():
    """Modelo de planejamento de geração de energia"""
    model = ConcreteModel()
    
    # Conjuntos
    model.Generators = Set(initialize=['Coal', 'Gas', 'Nuclear', 'Wind'])
    model.TimePeriods = Set(initialize=range(1, 25))  # 24 horas
    
    # Parâmetros do primeiro estágio (investimento)
    model.CapacityCost = Param(model.Generators, initialize={
        'Coal': 1000, 'Gas': 800, 'Nuclear': 3000, 'Wind': 1200
    })
    model.MaxCapacity = Param(model.Generators, initialize={
        'Coal': 500, 'Gas': 400, 'Nuclear': 300, 'Wind': 200
    })
    
    # Variáveis do primeiro estágio (investimento)
    model.Capacity = Var(model.Generators, within=NonNegativeReals)
    
    # Parâmetros estocásticos (serão definidos por cenário)
    model.Demand = Param(model.TimePeriods, initialize=0, mutable=True)
    model.WindAvailability = Param(model.TimePeriods, initialize=0, mutable=True)
    model.FuelCost = Param(model.Generators, initialize=0, mutable=True)
    
    # Restrições do primeiro estágio
    def capacity_rule(model, g):
        return model.Capacity[g] <= model.MaxCapacity[g]
    model.CapacityLimit = Constraint(model.Generators, rule=capacity_rule)
    
    # Função objetivo do primeiro estágio (custo de investimento)
    model.FirstStageCost = Expression(
        expr = sum(model.CapacityCost[g] * model.Capacity[g] 
                  for g in model.Generators)
    )
    
    return model

def create_scenario_instance(model, scenario_data):
    """Cria instância de cenário específica"""
    instance = model.clone()
    
    # Adicionar variáveis do segundo estágio
    instance.Generation = Var(
        instance.Generators, 
        instance.TimePeriods,
        within=NonNegativeReals
    )
    
    # Definir parâmetros estocásticos
    for t in instance.TimePeriods:
        instance.Demand[t] = scenario_data['demand'][t-1]
        instance.WindAvailability[t] = scenario_data['wind'][t-1]
    
    for g in instance.Generators:
        instance.FuelCost[g] = scenario_data['fuel_cost'][g]
    
    # Restrições do segundo estágio
    def demand_satisfaction_rule(model, t):
        return sum(model.Generation[g, t] for g in model.Generators) \
               >= model.Demand[t]
    instance.DemandSatisfaction = Constraint(
        instance.TimePeriods, 
        rule=demand_satisfaction_rule
    )
    
    def capacity_utilization_rule(model, g, t):
        if g == 'Wind':
            return model.Generation[g, t] <= \
                   model.Capacity[g] * model.WindAvailability[t]
        else:
            return model.Generation[g, t] <= model.Capacity[g]
    instance.CapacityUtilization = Constraint(
        instance.Generators,
        instance.TimePeriods,
        rule=capacity_utilization_rule
    )
    
    # Função objetivo do segundo estágio (custo operacional)
    def second_stage_cost_rule(model):
        return sum(model.FuelCost[g] * model.Generation[g, t] 
                  for g in model.Generators for t in model.TimePeriods)
    instance.SecondStageCost = Expression(rule=second_stage_cost_rule)
    
    return instance

# Gerar dados de cenários
def generate_energy_scenarios(num_scenarios=20):
    scenarios = []
    
    for s in range(num_scenarios):
        # Gerar demanda aleatória (padrão diário + ruído)
        base_demand = [50 + 30*np.sin(2*np.pi*t/24) for t in range(24)]
        demand = [d * np.random.lognormal(mean=0, sigma=0.1) 
                 for d in base_demand]
        
        # Disponibilidade eólica
        wind = [0.7 + 0.3*np.sin(2*np.pi*t/24 + np.pi/2) 
               * np.random.beta(a=2, b=2) for t in range(24)]
        
        # Custos de combustível
        fuel_cost = {
            'Coal': np.random.uniform(20, 30),
            'Gas': np.random.uniform(40, 60),
            'Nuclear': np.random.uniform(5, 10),
            'Wind': 0.0
        }
        
        scenario_data = {
            'demand': demand,
            'wind': wind,
            'fuel_cost': fuel_cost,
            'probability': 1.0/num_scenarios
        }
        scenarios.append(scenario_data)
    
    return scenarios

# Script principal usando PySP
if __name__ == '__main__':
    import pyomo.pysp.ph as ph
    
    # Configurar diretórios do PySP
    import os
    os.makedirs('ScenarioData', exist_ok=True)
    
    # Gerar cenários e salvar em arquivos
    scenarios = generate_energy_scenarios(10)
    
    for i, scenario_data in enumerate(scenarios):
        with open(f'ScenarioData/Scenario{i+1}.dat', 'w') as f:
            f.write(f"# Scenario {i+1}\n")
            f.write(f"param probability := {scenario_data['probability']};\n")
            
            # Salvar demanda
            f.write("param Demand :=\n")
            for t, d in enumerate(scenario_data['demand'], 1):
                f.write(f"{t} {d}\n")
            f.write(";\n")
            
            # Salvar disponibilidade eólica
            f.write("param WindAvailability :=\n")
            for t, w in enumerate(scenario_data['wind'], 1):
                f.write(f"{t} {w}\n")
            f.write(";\n")
    
    # Criar arquivo de estrutura de cenários
    with open('ScenarioStructure.dat', 'w') as f:
        f.write("""
set Stages := FirstStage SecondStage;
set Nodes := RootNode;
        
param NodeStage :=
    RootNode    FirstStage;
        
set Children[RootNode] := ;
        
param ConditionalProbability := ;
        
set Scenarios := """ + 
' '.join([f'Scenario{i+1}' for i in range(len(scenarios))]) + 
""";
        
param ScenarioLeafNode :=
""" + 
'\n'.join([f'    Scenario{i+1}   RootNode' for i in range(len(scenarios))]) +
""";
        
param StageCostVariable :=
    FirstStage    FirstStageCost
    SecondStage   SecondStageCost;
        """)
    
    # Executar Progressive Hedging
    from pyomo.pysp.phsolverserver import PHSolverServer
    
    options = {
        '--solver': 'ipopt',
        '--max-iterations': 100,
        '--default-rho': 100.0,
        '--enable-progressive-termination': True,
        '--output-solver-log': False,
    }
    
    with PHSolverServer(options=options) as ph_server:
        results = ph_server.solve()
        
        print("\n" + "="*50)
        print("RESULTADOS DO PLANEJAMENTO ENERGÉTICO")
        print("="*50)
        
        # Extrair solução do primeiro estágio
        first_stage_solution = results.xstar
        
        print("\nCapacidades ótimas a instalar:")
        for gen in ['Coal', 'Gas', 'Nuclear', 'Wind']:
            capacity = first_stage_solution.get(f'Capacity[{gen}]', 0)
            print(f"  {gen}: {capacity:.2f} MW")
        
        print(f"\nCusto total esperado: ${results.objval:,.2f}")
```

### Exemplo 3: Gestão de Portfólio Financeiro
```python
# portfolio_optimization.py
from pyomo.environ import *
from pyomo.pysp import *
import numpy as np

def create_portfolio_model():
    """Modelo de otimização de portfólio com retornos incertos"""
    model = ConcreteModel()
    
    # Ativos
    model.Assets = Set(initialize=['Stocks', 'Bonds', 'Gold', 'RealEstate'])
    
    # Parâmetros do primeiro estágio
    model.InitialWealth = Param(initialize=1000000)
    model.TransactionCost = Param(initialize=0.01)  # 1%
    
    # Variáveis do primeiro estágio (alocação inicial)
    model.InitialAllocation = Var(model.Assets, within=NonNegativeReals)
    
    # Restrições do primeiro estágio
    def initial_budget_rule(model):
        return sum(model.InitialAllocation[a] for a in model.Assets) \
               == model.InitialWealth
    model.InitialBudget = Constraint(rule=initial_budget_rule)
    
    # Parâmetros estocásticos
    model.Return = Param(model.Assets, initialize=0, mutable=True)
    model.RiskFreeRate = Param(initialize=0, mutable=True)
    
    # Função objetivo do primeiro estágio (custos de transação)
    model.FirstStageCost = Expression(
        expr = model.TransactionCost * sum(
            model.InitialAllocation[a] for a in model.Assets
        )
    )
    
    return model

def generate_financial_scenarios(num_scenarios=50, num_periods=12):
    """Gerar cenários de retornos financeiros usando GBM"""
    from scipy.stats import norm
    
    scenarios = []
    
    # Parâmetros dos ativos
    asset_params = {
        'Stocks': {'mu': 0.08, 'sigma': 0.20},
        'Bonds': {'mu': 0.04, 'sigma': 0.05},
        'Gold': {'mu': 0.03, 'sigma': 0.15},
        'RealEstate': {'mu': 0.06, 'sigma': 0.10}
    }
    
    for s in range(num_scenarios):
        returns = {}
        
        for asset, params in asset_params.items():
            # Simular retornos mensais
            monthly_mu = params['mu'] / 12
            monthly_sigma = params['sigma'] / np.sqrt(12)
            
            asset_returns = []
            for t in range(num_periods):
                ret = np.random.lognormal(
                    mean=monthly_mu,
                    sigma=monthly_sigma
                ) - 1
                asset_returns.append(ret)
            
            returns[asset] = asset_returns
        
        # Taxa livre de risco
        risk_free = [0.03/12] * num_periods
        
        scenario_data = {
            'returns': returns,
            'risk_free': risk_free,
            'probability': 1.0/num_scenarios
        }
        
        scenarios.append(scenario_data)
    
    return scenarios

# Implementação usando L-Shaped method para programação linear
def run_lshaped_portfolio():
    from pyomo.pysp.lshaped import LShapedMethod
    
    # Configurar opções
    options = {
        '--solver': 'cplex',
        '--max-iterations': 100,
        '--tee': True,
        '--check-bound-improvement-frequency': 10,
        '--linearize-nonbinary-penalty-terms': True,
    }
    
    # Criar e executar L-Shaped
    lshaped = LShapedMethod(options=options)
    
    # Configurar callbacks para controle personalizado
    class PortfolioCallback:
        def __init__(self):
            self.iterations = []
        
        def post_iteration(self, ph, scenario_tree, results):
            iteration = len(self.iterations) + 1
            bound = results['bound']
            objective = results['objective']
            
            self.iterations.append({
                'iteration': iteration,
                'lower_bound': bound,
                'objective': objective
            })
            
            print(f"Iteração {iteration}: "
                  f"LB={bound:.2f}, Obj={objective:.2f}")
    
    callback = PortfolioCallback()
    lshaped.callback = callback
    
    # Resolver
    results = lshaped.solve()
    
    return results, callback.iterations
```

## 7. Tipos de Problemas Específicos

### 7.1 Chance-Constrained Programming
```python
# chance_constrained.py
from pyomo.environ import *
from pyomo.pysp.ccutils import *

def create_chance_constrained_model():
    model = ConcreteModel()
    
    model.x = Var(within=NonNegativeReals)
    model.y = Var(within=NonNegativeReals)
    
    # Parâmetro estocástico
    model.uncertain_param = Param(initialize=0, mutable=True)
    
    # Restrição chance-constrained
    # P(ax + by >= c) >= 0.95
    model.chance_constraint = StochasticConstraint(
        rule=lambda m: m.x + m.uncertain_param * m.y >= 10
    )
    
    # Configurar chance constraint
    model.chance_constraint.probability = 0.95
    model.chance_constraint.violation_cost = 1000
    
    return model
```

### 7.2 Multi-stage Stochastic Programming
```python
# multistage.py
from pyomo.environ import *
from pyomo.pysp.scenariotree import *

def create_multistage_model(num_stages=3):
    model = ConcreteModel()
    
    model.Stages = RangeSet(1, num_stages)
    
    # Variáveis por estágio
    model.x = Var(model.Stages, within=NonNegativeReals)
    
    # Parâmetros estocásticos
    model.demand = Param(model.Stages, initialize=0, mutable=True)
    model.price = Param(model.Stages, initialize=0, mutable=True)
    
    # Restrições de não-antecipação são automaticamente
    # impostas pela estrutura da árvore de cenários
    
    def objective_rule(model):
        return sum(model.price[t] * model.x[t] for t in model.Stages)
    
    model.obj = Objective(rule=objective_rule, sense=maximize)
    
    return model

# Criar árvore de cenários multi-estágio
def create_multistage_scenario_tree():
    from pyomo.pysp.scenariotree.tree_structure_model import \
        CreateConcreteScenarioTreeModel
    
    tree_model = CreateConcreteScenarioTreeModel()
    
    # Definir estágios
    tree_model.Stages.add('Stage1')
    tree_model.Stages.add('Stage2')
    tree_model.Stages.add('Stage3')
    
    # Definir nós
    tree_model.Nodes.add('Root')
    
    # Adicionar nós do segundo estágio
    for i in range(3):
        node_name = f'Stage2_Node{i}'
        tree_model.Nodes.add(node_name)
        tree_model.NodeStage[node_name] = 'Stage2'
        tree_model.Children['Root'].add(node_name)
        tree_model.ConditionalProbability[node_name] = 1.0/3
    
    # Adicionar nós do terceiro estágio
    for i in range(3):
        parent = f'Stage2_Node{i}'
        for j in range(2):
            node_name = f'Stage3_Node{i}_{j}'
            tree_model.Nodes.add(node_name)
            tree_model.NodeStage[node_name] = 'Stage3'
            tree_model.Children[parent].add(node_name)
            tree_model.ConditionalProbability[node_name] = 0.5
    
    return tree_model
```

## 8. Boas Práticas e Otimização

### 8.1 Escolha do Algoritmo
```python
ALGORITHM_CHOICE = {
    'linear_2stage': 'LShapedMethod',
    'linear_multistage': 'ProgressiveHedging',
    'nonlinear': 'ProgressiveHedging',
    'integer': 'ProgressiveHedging',
    'chance_constrained': 'SampleAverageApproximation',
    'robust': 'RobustOptimizationModule'
}
```

### 8.2 Controle de Convergência
```python
def adaptive_rho_control(ph, iteration, gap_history):
    """Controle adaptativo do parâmetro rho no PH"""
    
    if iteration < 10:
        return 1.0  # Valor inicial
    
    # Calcular taxa de convergência
    recent_gaps = gap_history[-5:]
    if len(recent_gaps) >= 2:
        convergence_rate = recent_gaps[-1] / recent_gaps[0]
        
        if convergence_rate > 0.9:  # Convergência lenta
            return ph.default_rho * 1.5
        elif convergence_rate < 0.5:  # Oscilando
            return ph.default_rho * 0.7
    
    return ph.default_rho
```

### 8.3 Redução de Cenários
```python
from pyomo.pysp.scenarioreduction import *

def reduce_scenarios(scenarios, target_num=10, method='fast_forward'):
    """
    Reduz número de cenários mantendo propriedades estatísticas
    """
    
    if method == 'fast_forward':
        reducer = FastForwardSelector()
    elif method == 'backward':
        reducer = BackwardSelector()
    elif method == 'kmeans':
        reducer = KMeansSelector(n_clusters=target_num)
    else:
        reducer = RandomSelector()
    
    reduced = reducer.select(scenarios, target_num)
    return reduced
```

## 9. Integração com Outras Bibliotecas

### 9.1 PySP + Pandas para Análise de Resultados
```python
import pandas as pd
from pyomo.pysp import *

def analyze_results(ph_results):
    """Analisar resultados do PH usando pandas"""
    
    # Extrair soluções por cenário
    scenario_solutions = []
    for scenario_name, scenario_solution in ph_results.scenario_solutions.items():
        scenario_data = {
            'scenario': scenario_name,
            'probability': ph_results.scenario_probabilities[scenario_name],
            'objective': scenario_solution.objective
        }
        
        # Adicionar variáveis
        for var_name, var_value in scenario_solution.variables.items():
            scenario_data[var_name] = var_value
        
        scenario_solutions.append(scenario_data)
    
    # Criar DataFrame
    df = pd.DataFrame(scenario_solutions)
    
    # Análise estatística
    summary = {
        'expected_value': (df['objective'] * df['probability']).sum(),
        'std_deviation': df['objective'].std(),
        'cvar_95': calculate_cvar(df, alpha=0.95),
        'best_case': df['objective'].max(),
        'worst_case': df['objective'].min()
    }
    
    return df, summary

def calculate_cvar(df, alpha=0.95):
    """Calcular Conditional Value at Risk"""
    sorted_obj = df['objective'].sort_values()
    n = len(sorted_obj)
    k = int((1-alpha) * n)
    
    if k == 0:
        return sorted_obj.iloc[0]
    
    return sorted_obj.iloc[:k].mean()
```

### 9.2 PySP + Scikit-learn para Geração de Cenários
```python
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

def generate_scenarios_gp(historical_data, num_scenarios=100):
    """Gerar cenários usando Gaussian Process"""
    
    X = historical_data[['time', 'features']].values
    y = historical_data['target'].values
    
    # Treinar GP
    kernel = RBF(length_scale=1.0) + WhiteKernel(noise_level=1)
    gp = GaussianProcessRegressor(kernel=kernel)
    gp.fit(X, y)
    
    # Gerar cenários
    scenarios = []
    for _ in range(num_scenarios):
        # Amostrar do processo gaussiano
        y_sample, y_std = gp.sample_y(X, random_state=None)
        
        scenario = {
            'values': y_sample.flatten(),
            'probability': 1.0/num_scenarios
        }
        scenarios.append(scenario)
    
    return scenarios
```

## 10. Casos de Uso Reais

### 10.1 Gestão de Recursos Hídricos
```python
# water_management.py
class WaterReservoirManagement:
    """Gestão de reservatórios com afluências incertas"""
    
    def __init__(self, num_reservoirs, num_periods):
        self.model = self._create_model(num_reservoirs, num_periods)
    
    def _create_model(self, num_reservoirs, num_periods):
        model = ConcreteModel()
        
        model.Reservoirs = RangeSet(1, num_reservoirs)
        model.Periods = RangeSet(1, num_periods)
        
        # Estado do reservatório (volume armazenado)
        model.Storage = Var(model.Reservoirs, model.Periods, 
                           within=NonNegativeReals)
        
        # Decisões: liberação, geração, spill
        model.Release = Var(model.Reservoirs, model.Periods,
                           within=NonNegativeReals)
        model.Generate = Var(model.Reservoirs, model.Periods,
                            within=NonNegativeReals)
        
        # Parâmetros estocásticos
        model.Inflow = Param(model.Reservoirs, model.Periods,
                            initialize=0, mutable=True)
        
        # Balanço hídrico
        def water_balance_rule(model, r, t):
            if t == 1:
                return model.Storage[r,t] == model.Storage0[r] \
                       + model.Inflow[r,t] - model.Release[r,t]
            else:
                return model.Storage[r,t] == model.Storage[r,t-1] \
                       + model.Inflow[r,t] - model.Release[r,t]
        
        model.WaterBalance = Constraint(model.Reservoirs, model.Periods,
                                        rule=water_balance_rule)
        
        return model
```

### 10.2 Logística sob Demanda Incerta
```python
# logistics.py
class StochasticSupplyChain:
    """Cadeia de suprimentos com demanda incerta"""
    
    def __init__(self, locations, products, periods):
        self.model = self._create_model(locations, products, periods)
    
    def _create_model(self, locations, products, periods):
        model = ConcreteModel()
        
        model.Locations = Set(initialize=locations)
        model.Products = Set(initialize=products)
        model.Periods = Set(initialize=periods)
        
        # Decisões estratégicas (primeiro estágio)
        model.WarehouseOpen = Var(model.Locations, within=Binary)
        model.TransportCapacity = Var(model.Locations, model.Locations,
                                     within=NonNegativeReals)
        
        # Parâmetros estocásticos
        model.Demand = Param(model.Locations, model.Products, model.Periods,
                            initialize=0, mutable=True)
        
        # Variáveis operacionais (segundo estágio)
        model.Transport = Var(model.Locations, model.Locations,
                             model.Products, model.Periods,
                             within=NonNegativeReals)
        
        model.Inventory = Var(model.Locations, model.Products, model.Periods,
                             within=NonNegativeReals)
        
        # Restrições de capacidade (vinculando estágios)
        def capacity_linking_rule(model, i, j, p, t):
            return model.Transport[i,j,p,t] <= model.TransportCapacity[i,j]
        
        model.CapacityLinking = Constraint(
            model.Locations, model.Locations,
            model.Products, model.Periods,
            rule=capacity_linking_rule
        )
        
        return model
```

## 11. Comparação com Outras Abordagens

```python
COMPARISON_TABLE = {
    'PySP': {
        'strengths': [
            'Integração nativa com Pyomo',
            'Suporte a múltiplos algoritmos',
            'Estrutura de cenários flexível',
            'Boa documentação e comunidade'
        ],
        'weaknesses': [
            'Curva de aprendizado íngreme',
            'Performance para muitos cenários',
            'Limitado a decomposição'
        ],
        'best_for': [
            'Problemas de médio porte',
            'Quando precisa de flexibilidade',
            'Integração com ecossistema Pyomo'
        ]
    },
    'Alternative': {
        'SHOT': 'Para MINLP estocástico',
        'ROC++': 'Para robust optimization',
        'SPInE': 'Para ensino e prototipagem',
        'Commercial': 'GAMS, AIMMS, Xpress'
    }
}
```

## 12. Conclusão

**PySP** é uma ferramenta poderosa para **Programação Estocástica** dentro do ecossistema Pyomo. Suas principais vantagens são:

### Pontos Fortes:
1. **Integração perfeita** com modelos Pyomo existentes
2. **Múltiplos algoritmos** de decomposição
3. **Flexibilidade** na definição de cenários
4. **Ativo desenvolvimento** e comunidade

### Aplicações Típicas:
- **Finance**: Portfolio optimization under uncertainty
- **Energy**: Power generation planning with renewable variability
- **Logistics**: Supply chain design with uncertain demand
- **Water**: Reservoir management with stochastic inflows
- **Agriculture**: Crop planning with weather uncertainty

### Quando usar PySP vs outras abordagens:
- Use **PySP** quando já trabalha com Pyomo e precisa de estocasticidade
- Use **amostragem direta** para problemas pequenos com poucos cenários
- Use **comerciais** (GAMS, AIMMS) para problemas muito grandes em produção
- Use **metodologias específicas** (RO, DRO) para incerteza estruturada

### Próximos Passos:
1. Comece com problemas **two-stage linear**
2. Experimente diferentes **algoritmos de decomposição**
3. Otimize **parâmetros do algoritmo** (como ρ no PH)
4. Implemente **redução de cenários** para problemas grandes
5. Explore **integração com machine learning** para geração de cenários

PySP transforma problemas estocásticos complexos em modelos gerenciáveis, permitindo que você foque na modelagem do problema enquanto a biblioteca cuida da complexidade computacional da decomposição estocástica.